In [2]:
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.8 MB/s eta 0:00:00


In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error

sim = AerSimulator()

print("Qiskit is ready!")

Qiskit is ready!


In [5]:
# Exercise 1: Implement the superdense coding circuit for the message '00' and verify correct decoding.

qc = QuantumCircuit(2, 2)

# Create shared entangled Bell pair
qc.h(0)
qc.cx(0, 1)

# Alice encodes message 00
# No gate is required

# Bob decodes the message
qc.cx(0, 1)
qc.h(0)

# Measure the qubits
qc.measure([0, 1], [0, 1])

result = sim.run(qc, shots=1024).result()
print("Message 00:", result.get_counts())

Message 00: {'00': 1024}


In [6]:
# Exercise 2: Implement superdense coding for all four possible 2-bit messages ('00', '01', '10', '11') and tabulate the decoded results.

messages = ["00", "01", "10", "11"]

for message in messages:

    qc = QuantumCircuit(2, 2)

    # Create shared entangled Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Alice encodes the message
    if message == "01":
        qc.x(0)

    elif message == "10":
        qc.z(0)

    elif message == "11":
        qc.x(0)
        qc.z(0)

    # Bob decodes
    qc.cx(0, 1)
    qc.h(0)

    # Measurement
    qc.measure([0, 1], [0, 1])

    result = sim.run(qc, shots=1024).result()
    print(message, "->", result.get_counts())

00 -> {'00': 1024}
01 -> {'10': 1024}
10 -> {'01': 1024}
11 -> {'11': 1024}


In [7]:
# Exercise 3: Introduce a bit-flip error on the transmitted qubit (simulate channel noise) and analyse the impact on decoding accuracy.

qc = QuantumCircuit(2, 2)

# Create shared Bell pair
qc.h(0)
qc.cx(0, 1)

# Alice encodes message 00

# Introduce bit-flip error
qc.x(0)

# Bob decodes
qc.cx(0, 1)
qc.h(0)

# Measurement
qc.measure([0, 1], [0, 1])

result = sim.run(qc, shots=1024).result()
print("With bit-flip error:", result.get_counts())

With bit-flip error: {'10': 1024}


In [8]:
# Exercise 4: Compare the classical bandwidth requirement versus the superdense-coded transmission for a hypothetical 100-message communication session.

messages = 100

# Classical communication
classical_bits = messages * 2

# Superdense coding
quantum_qubits = messages

print("Number of messages:", messages)
print("Classical bits required:", classical_bits)
print("Qubits transmitted using superdense coding:", quantum_qubits)

Number of messages: 100
Classical bits required: 200
Qubits transmitted using superdense coding: 100


In [9]:
# Exercise 5: Extend the protocol to transmit 2 classical bits using a noisy entangled pair with a specified fidelity, and estimate the resulting decoding error rate through Qiskit noise simulation.

noise = NoiseModel()

error_probability = 0.10

bit_flip = pauli_error([
    ('X', error_probability),
    ('I', 1 - error_probability)
])

noise.add_all_qubit_quantum_error(bit_flip, ['x'])

correct = 0
shots = 1024

for i in range(shots):

    qc = QuantumCircuit(2, 2)

    # Create shared Bell pair
    qc.h(0)
    qc.cx(0, 1)

    # Alice encodes message 00

    # Bob decodes
    qc.cx(0, 1)
    qc.h(0)

    # Measure
    qc.measure([0, 1], [0, 1])

    result = sim.run(
        qc,
        shots=1,
        noise_model=noise
    ).result()

    counts = result.get_counts()

    if "00" in counts:
        correct += 1

accuracy = correct / shots * 100
error_rate = 100 - accuracy

print("Decoding Accuracy:", round(accuracy, 2), "%")
print("Decoding Error Rate:", round(error_rate, 2), "%")

Decoding Accuracy: 100.0 %
Decoding Error Rate: 0.0 %
